# What makes an LLM *agentic*?

An LLM is just **text in → text out**. It cannot run code, look things up, or act on the world by itself.

What makes it **agentic** is the *harness* around it: a loop that

1. sends the current context to the model,
2. the model decides the **next step** (call a tool, or give a final answer),
3. the harness **executes** that step,
4. feeds the result back, and repeats until done.

```
observation → model decides → harness acts → result fed back → repeat
```

Below we give the model a single tool (`calculate`) and two problems:

- **Task 1 — a chain of calculations.** Each arithmetic step needs its own tool call, so you can watch the loop iterate.
- **Task 2 — self-verify.** The agent must re-check its own answer (divide back) before it may stop: a first taste of *reflection*.


In [ ]:
import ast
import json
import operator
import time

import ollama

MODEL = "qwen3.5:2b"
MAX_STEPS = 10
MAX_TOKENS = 500
client = ollama.Client(timeout=300.0)



In [ ]:
# --- Tools -------------------------------------------------------------------
# An LLM is just text -> text. It cannot run code or do arithmetic by itself.
# "Tools" are plain Python functions that the *harness* (not the model) executes.
ALLOWED_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.FloorDiv: operator.floordiv,
}


def calculate(expression: str = None, expr: str = None,
              operation: str = None, value: str = None, op: str = None) -> str:
    """Evaluate ONE simple arithmetic operation and return the result.

    Call with args {"expression": "137 * 289"}. Only a single binary operation
    is accepted; for a chain you must call this tool once per operation, waiting
    for each result before issuing the next step.
    """
    expression = expression or expr or operation or value or op
    tree = ast.parse(expression, mode="eval")
    allowed_nodes = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant,
                     ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Mod, ast.Pow,
                     ast.FloorDiv, ast.USub, ast.UAdd)
    if any(not isinstance(node, allowed_nodes) for node in ast.walk(tree)):
        raise ValueError(f"unsupported expression: {expression!r}")
    operators = [n for n in ast.walk(tree) if isinstance(n, ast.BinOp)]
    if len(operators) != 1:
        raise ValueError(
            f"Only ONE operation per call is allowed (got {len(operators)} in "
            f"{expression!r}). Split the chain into single steps."
        )
    return f"{_eval(tree.body)}"


def _eval(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return ALLOWED_OPS[type(node.op)](_eval(node.left), _eval(node.right))
    if isinstance(node, ast.UnaryOp):
        return ALLOWED_OPS[type(node.op)](_eval(node.operand))
    raise ValueError(f"unsupported node: {node!r}")


TOOLS = {"calculate": calculate}

TOOL_DESCRIPTIONS = "\n".join(
    f"- {name}: {fn.__doc__}" for name, fn in TOOLS.items()
)



In [ ]:
def _extract_json(text: str):
    """Parse a JSON object out of a model reply, tolerating markdown fences."""
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.lower().startswith("json"):
            text = text[4:]
        text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end > start:
        return json.loads(text[start:end + 1])
    raise json.JSONDecodeError("no JSON object found", text, 0)


SYSTEM_PROMPT = f"""You are a helpful agent. You can use the following tools:
{TOOL_DESCRIPTIONS}

You MUST reply with ONLY one valid JSON object and nothing else, in one of two forms:

1. To use a tool:
{{"thought": "<why you are taking this step>", "tool": "calculate", "args": {{"expression": "23 * 17"}}}}

2. When you have the final answer:
{{"thought": "<why you are done>", "final_answer": "<your answer>"}}

RULES:
- Never do arithmetic in your head. For EVERY arithmetic operation you must call the
  calculate tool, one operation at a time. Wait for the tool result before the next step.
- The calculate tool takes a single operation via the "expression" argument, e.g. "137 * 289".
- Always write a short "thought" explaining what you are doing and why.
  Keep it to ONE short sentence.
- Only give a final_answer once every needed calculation has been done via the tool.
- The final_answer number must EXACTLY match the result of your last calculate call.
  Do not retype it from memory.
- If asked to verify or double-check your answer, use the inverse operation with the
  calculate tool (e.g. after multiplying, divide back) and confirm the result before
  you give the final_answer.

Do not add any text outside the JSON."""


def _chat(model, messages):
    """Chat with retries — local models can intermittently stall or time out."""
    last_exc = None
    for attempt in range(3):
        try:
            return client.chat(model=model, messages=messages, think=False,
                               format="json",
                               options={"num_predict": MAX_TOKENS})
        except Exception as exc:
            last_exc = exc
            time.sleep(2)
    raise last_exc




In [ ]:
# --- The agentic harness ------------------------------------------------------
# The "agency" lives here, in this loop — NOT inside the model. The model only
# proposes the next step; the harness executes it and feeds the result back.
def run_agent(question: str, verbose: bool = True) -> str:
    history = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    tools_used = 0

    for step in range(1, MAX_STEPS + 1):
        response = _chat(MODEL, history)
        content = response["message"]["content"]

        try:
            parsed = _extract_json(content)
        except json.JSONDecodeError:
            # Model returned non-JSON: nudge it back into the loop.
            if verbose:
                print(f"{'─' * 8} Iteration {step} {'─' * 30}")
                print(f"Thought:     (invalid reply — re-prompting)")
                print(f"Observation: {content[:80]!r}")
            history.append(
                {"role": "user",
                 "content": "Your reply was not valid JSON. Reply with ONLY one JSON object."}
            )
            continue

        if "final_answer" in parsed:
            if tools_used == 0:
                # Model tried to answer without using any tool: force the loop.
                if verbose:
                    print(f"{'─' * 8} Iteration {step} {'─' * 30}")
                    print(f"Thought:     {parsed.get('thought', '')}")
                    print("Observation: (tried to answer before using any tool — re-prompting)")
                history.append(
                    {"role": "user",
                     "content": "Do not answer from memory. You must first call the "
                                "calculate tool to compute the answer, then base "
                                "final_answer on the tool result."}
                )
                continue
            if verbose:
                print(f"{'─' * 8} Iteration {step} {'─' * 30}")
                print(f"Thought:  {parsed.get('thought', '')}")
                print(f"Answer:   {parsed['final_answer']}")
            return parsed["final_answer"]

        tool_name = parsed.get("tool")
        if isinstance(tool_name, str) and tool_name in TOOLS:
            args = parsed.get("args", {})
            try:
                tool_result = TOOLS[tool_name](**args)
            except Exception as exc:
                tool_result = f"Error: {exc}"
            tools_used += 1
            if verbose:
                print(f"{'─' * 8} Iteration {step} {'─' * 30}")
                print(f"Thought:     {parsed.get('thought', '')}")
                print(f"Action:      {tool_name}({args!r})")
                print(f"Observation: {tool_result}")
            history.append(
                {"role": "user", "content": f"Tool result: {tool_result}"}
            )
            continue

        if verbose:
            print(f"{'─' * 8} Iteration {step} {'─' * 30}")
            print(f"Thought:     {parsed.get('thought', '')}")
            print(f"Observation: (unknown tool / malformed reply)")
        history.append(
            {"role": "user",
             "content": f"Unknown tool or malformed reply: {content!r}"}
        )

    return "Agent reached the step limit without a final answer."




In [ ]:
chain_question = ("Start with 137, multiply by 289, then divide that result by 17, "
                  "then add 23, and give me the final answer.")

print("=" * 70)
print("TASK 1 — CHAIN OF CALCULATIONS  (one tool call per operation)")
print("=" * 70)
answer = run_agent(chain_question)
print("-" * 70)
print(f"FINAL ANSWER: {answer}")


In [ ]:
verify_question = ("Multiply 456 by 789, then verify your answer by dividing the "
                   "result by 789 and confirming you get 456 back. "
                   "Give me the verified final answer.")

print("=" * 70)
print("TASK 2 — SELF-VERIFY  (agent re-checks its own answer before finalising)")
print("=" * 70)
answer = run_agent(verify_question)
print("-" * 70)
print(f"FINAL ANSWER: {answer}")


In [ ]:
# --- The non-agentic contrast --------------------------------------------------
# No loop, no tools: the model is asked once. It either refuses or hallucinates,
# which shows it's the loop (not the model) that provides agency.
def run_naive(question: str) -> str:
    response = client.chat(
        model=MODEL,
        messages=[{"role": "user", "content": question}],
        think=False,
        options={"num_predict": MAX_TOKENS},
    )
    return response["message"]["content"]



print("=" * 70)
print("NAIVE RUN  (single shot, no loop, no tools)")
print("=" * 70)
print(run_naive(chain_question))


## Takeaway

Compare the runs:

- **Agentic run (Task 1)** — the model is inside a loop. Each iteration you see the full *ReAct* cycle:
  `Thought → Action → Observation`. The model explains what it's doing (`thought`), the harness runs the
  `calculate` action, and the result (`observation`) is fed back. Because `calculate` only accepts *one*
  operation at a time, the model must iterate once per arithmetic step.
- **Agentic run (Task 2)** — reflection: the model verifies its own answer with the inverse operation
  before it is allowed to stop. The loop drives it to *doubt and confirm*, not just produce an answer.
- **Naive run** — a single `text → text` call with no loop and no tools. The model has no way to act, so it guesses.

The model is the same in both cases. What changed is the **harness** around it.
